# Import Library

In [ ]:
import pandas as pd
import nltk
import pickle
import os
import string

from nltk.tokenize import word_tokenize
from nltk import FreqDist
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tag import pos_tag
from nltk.corpus import wordnet, stopwords
from random import shuffle

# Download NLTK Library

In [29]:
nltk.download("wordnet")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")
nltk.download("punkt")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

# Settings Variable

In [30]:
stemmer = PorterStemmer()
wnl = WordNetLemmatizer()
eng_stopwords = stopwords.words("english")

# Preprocessing

In [31]:
def preprocessing(document):
    words = word_tokenize(document.lower())
    words = [wnl.lemmatize(word) for word in words]
    words = [stemmer.stem(word) for word in words]
    return {word : True for word in words if word not in eng_stopwords and word.isalpha()}

# Training Model

In [ ]:
def train_model():

    # Read Data
    data = pd.read_csv("./Dataset/Tweets.csv")

    # Feature Sets
    feature_sets = [(preprocessing(text), label) for text, label in zip (data["text"], data["airline_sentiment"])]

    split_index = int(len(feature_sets) * 0.85)

    train_set, test_set = feature_sets[:split_index], feature_sets[split_index:]

    classifier = nltk.NaiveBayesClassifier.train(train_set)
    accuracy = nltk.classify.accuracy(classifier, test_set)

    print (f"Accuracy -> {accuracy}")
    classifier.show_most_informative_features(5)

    file = open("./model.pickle", "wb")
    pickle.dump(classifier, file)
    file.close()

    return classifier

def read_model():
    if os.path.exists("./model.pickle"):
        file = open("./model.pickle", "rb")
        classifier = pickle.load(file)
        file.close()
        print ("Model loaded successfully")
        classifier.show_most_informative_features(5)
    else:
        print ("Model not found. Train the model...")
        classifier = train_model()
    return classifier

# Functions

In [ ]:
def write_review():
    while True:
        review = input("Please input your review [>= 2 words]")
        words = review.split()
        if len(words) <= 1:
            print  ("Review must be more than 1 word")
        else:
            print ("Review added")
            break
    return review

def analyze_review(review, classifier):

    if len(review) == 0:
        print ("Please add review first")
        return
    
    words = word_tokenize(review.lower())
    words = FreqDist[(word for word in words if word not in string.punctuation and word.isalpha())]
    tagged = pos_tag(words)

    for i, word in enumerate(tagged):
        print (f"{i+1}. {word[0]} -> {word[1]}")

    for word in words:

        print (f"Word -> {word}")

        synsets = wordnet.synsets(word)

        synonyms = []
        antonyms = []

        for synset in synsets:
            for lemma in synset.lemmas():
                synonyms.append(lemma.name())

                for antonym in lemma.antonyms():
                    antonyms.append(antonym.name())


        print ("Synonyms")
        if len(synonyms) == 0:
            print ("No synonyms detected")
        else:
            for syn in synonyms[:5]:
                print (f"(+) : {syn}")
        
        print ("Antonyms")
        if len(antonyms) == 0:
            print ("No antonyms detected")
        else:
            for ant in antonyms[:5]:
                print (f"(-) : {ant}")

        print("===========================")

    
    clean_review = [word for word in word_tokenize(review) if word not in eng_stopwords and word not in string.punctuation]
    clean_review = [wnl.lemmatize(stemmer.stem(word)) for word in clean_review]
    category = classifier.classify(FreqDist(clean_review))

    print (f"Your Review: {review}")
    print (f"Your Review Category: {category}")

# Main Functions

In [34]:
def main_menu():

    classifier = read_model()

    review = ""

    while True:

        print ("1. Write Tweet")
        print ("2. Analyze Tweet")
        print ("3. Exit")

        print (f"Your current Tweet: {review}")

        choice = input("Input menu choice")

        if choice == '1':
            review = write_review()
        elif choice == '2':
            analyze_review(review, classifier)
        elif choice == '3':
            print ("Thank you :)")
            break
        else:
            print ("Input must be between 1 - 3")

In [35]:
main_menu()

Most Informative Features
                outstand = True           positi : negati =     29.7 : 1.0
                passbook = True           positi : negati =     29.7 : 1.0
                 fantast = True           positi : negati =     28.7 : 1.0
                  beauti = True           positi : negati =     27.2 : 1.0
                 favorit = True           positi : negati =     27.2 : 1.0
Model Load Successfully
1. Write Tweet
2. Analyze Tweet
3. Exit
Your current Tweet: 
Please add review first
1. Write Tweet
2. Analyze Tweet
3. Exit
Your current Tweet: 
Review added
1. Write Tweet
2. Analyze Tweet
3. Exit
Your current Tweet: Hello World
1. hello -> NN
2. world -> NN
Synonyms
(+) : hello
(+) : hullo
(+) : hi
(+) : howdy
(+) : how-do-you-do
Antonyms
No antonyms detected
Synonyms
(+) : universe
(+) : existence
(+) : creation
(+) : world
(+) : cosmos
Antonyms
No antonyms detected
Your Review: Hello World
Your Review Category: negative
1. Write Tweet
2. Analyze Tweet
3. Exit
Your